# Day 45 · From text to training batches: the shift-by-one trick

**First-principles question:** How does raw text become training examples?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import urllib.request
import torch

def download(url, name):
    path = ROOT / "data" / "downloads" / name
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(url, path)
    return path.read_text(encoding="utf-8")

text = download("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
                "tinyshakespeare.txt")
print(f"{len(text):,} characters\n", text[:300])

## 1 · Characters → numbers → characters

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
# TODO: stoi (character → number) and itos (number → character) dictionaries
stoi, itos = {}, {}

def encode(s):
    # TODO: a list of numbers, one per character
    ...

def decode(ids):
    # TODO: the string back
    ...

print(vocab_size, encode("hii there"), decode(encode("hii there")))

## 2 · Train / validation split

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
# TODO: the first 90% is train_data, the rest val_data
train_data, val_data = None, None
print(len(train_data), len(val_data))

## 3 · Batches
x = B random pieces of length T. y = the same pieces shifted one character to the left.

In [ ]:
torch.manual_seed(1337)
BATCH, BLOCK = 4, 8

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - BLOCK, (BATCH,))
    # TODO: x = torch.stack of d[i:i + BLOCK] for i in ix; y = the same, starting one later
    ...

x, y = get_batch("train")
print(x.shape, y.shape)

## 4 · The 8 examples hidden in one row

In [ ]:
for t in range(BLOCK):
    context, target = x[0, :t + 1], y[0, t]
    print(f"when the input is {decode(context.tolist())!r} the target is {decode([target.item()])!r}")

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert decode(encode(text[:1000])) == text[:1000], "decode(encode(text)) must give the text back"
assert len(train_data) == int(0.9 * len(data)) and len(train_data) + len(val_data) == len(data)
assert x.shape == (BATCH, BLOCK) and y.shape == (BATCH, BLOCK)
assert torch.equal(x[:, 1:], y[:, :-1]), "y is x shifted by one"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")